In [ ]:
import os
import requests

# 비디오 ID와 API 키 설정
# video_id = "dQw4w9WgXcQ"  # 예시 비디오 ID
api_key = os.environ["YOUTUBE_API_KEY"]  # 당신의 API 키
handle = "@MBC_officialchannel"
max_results = 50
total = 300


In [ ]:
# 핸들을 사용하여 채널 정보 검색
search_url = f"https://www.googleapis.com/youtube/v3/search?part=snippet&type=channel&q={handle}&key={api_key}"
response = requests.get(search_url)
data = response.json()

# 채널 ID 추출
channel_id = data['items'][0]['snippet']['channelId']
print(f"Channel ID for {handle}: {channel_id}")

In [ ]:
import requests



# 200개의 비디오를 가져오기 위한 변수 초기화
video_ids = []
next_page_token = None

# 4번의 요청으로 200개의 비디오 ID를 가져옴
while len(video_ids) < total:
    search_url = f"https://www.googleapis.com/youtube/v3/search?key={api_key}&channelId={channel_id}&part=snippet&order=viewCount&type=video&maxResults={max_results}&pageToken={next_page_token or ''}"
    response = requests.get(search_url)
    video_data = response.json()

    video_ids.extend([item['id']['videoId'] for item in video_data['items']])

    # 다음 페이지 토큰이 없으면 종료
    next_page_token = video_data.get('nextPageToken')
    if not next_page_token:
        break

# 200개의 비디오 ID로 자르기
video_ids = video_ids[:total]


In [ ]:

import requests
import isodate
from datetime import timedelta

filtered_videos = []


for video_id in video_ids:
    video_details_url = f"https://www.googleapis.com/youtube/v3/videos?key={api_key}&id={video_id}&part=statistics,contentDetails,snippet"
    response = requests.get(video_details_url)
    details_data = response.json()

    # 'items' 키가 있는지 확인
    if 'items' not in details_data or len(details_data['items']) == 0:
        print(f"Error: No data for video ID {video_id}")
        continue

    item = details_data['items'][0]  # 첫 번째 항목 선택
    view_count = int(item['statistics']['viewCount'])

    # 동영상 길이 파싱
    duration_iso = item['contentDetails']['duration']
    duration = isodate.parse_duration(duration_iso)
    
    # 5분에서 20분 사이인지 확인
    if view_count >= 50000 and timedelta(minutes=5) <= duration <= timedelta(minutes=20):
        filtered_videos.append({
            'title': item['snippet']['title'],
            'videoId': item['id'],
            'viewCount': view_count,
            'duration': str(duration)  # 결과 출력 시 보기 좋게 문자열로 변환
        })

# 조회수 내림차순으로 정렬
filtered_videos = sorted(filtered_videos, key=lambda x: x['viewCount'], reverse=True)

# 결과 출력
for video in filtered_videos:
    print(f"Title: {video['title']}, Video ID: {video['videoId']}, View Count: {video['viewCount']}, Duration: {video['duration']}")

In [ ]:
import pickle

with open('filtered_videos.pkl', 'wb') as file:
    pickle.dump(filtered_videos, file)



In [ ]:
with open('filtered_videos.pkl', 'rb') as file:
    loaded_videos = pickle.load(file)

In [ ]:
video_ids = [video['videoId'] for video in loaded_videos]


In [ ]:
video_ids

In [ ]:
with open('video_ids.pkl', 'wb') as file:
    pickle.dump(video_ids, file)

In [ ]:
len(video_ids)